FastAPI 允许同时编写两种路由
```
@app.get("/async")
async def async_route():
    ...

@app.get("/sync")
def sync_route():
    ...
```

### async def 路由在哪里执行？
async def 路由直接运行在事件循环中。
```
@app.get("/users")
async def get_users():
    users = await async_database.fetch_all()
    return users
```
遇到await async_database.fetch_all()时，会暂停当前协程，将控制权交回事件循环，等待异步操作完成。

请求 A 发起数据库查询
→ await 暂停请求 A
→ 事件循环处理请求 B
→ 数据库结果返回
→ 恢复请求 A

就是异步 I/O 能够高效处理大量并发等待的原因。

### 普通 def 路由在哪里执行？

FastAPI 不会直接在事件循环中运行这个普通函数，而是把它放进**外部线程池**执行，并**异步等待结果**。
```
事件循环收到请求
→ 将 def 路由提交给线程池
→ 某个线程执行同步查询
→ 事件循环继续处理其他请求
→ 线程完成
→ FastAPI 获取结果并返回响应
```

因此，如果使用只能同步调用的数据库或 HTTP 客户端，可以用def 路由来执行。

> #### 同步路由不是“变成异步了”
> 线程池只是把阻塞从事件循环线程移到其他线程,线程中的同步查询仍然要等待，只是它不会占住事件循环.
> 线程池有代价：线程需要内存、存在线程切换成本、线程数量有限、大量阻塞任务会占满线程池、后续同步任务需要排队

FastAPI 使用有限的线程池隔离同步阻塞操作，保护事件循环，但线程池本身仍是一种有限资源。

**只有由 FastAPI 调用的路由和依赖会得到这种线程池处理；你自己调用的普通工具函数仍按普通 Python 调用规则直接执行。**

In [ ]:
def load_data():
    return sync_database.query()


@app.get("/data")
async def get_data():
    result = load_data()
    return result

"""
async 路由运行在事件循环
→ 直接调用 load_data()
→ load_data 也运行在事件循环线程
→ 同步查询阻塞事件循环
"""

#### 如何处理同步辅助函数？

In [ ]:
# 方式一：将整个路由写成 def
@app.get("/data")
def get_data():
    result = load_data()
    return result

# FastAPI 会把整个路由放进线程池。

适合：
- 路由主要由同步代码组成
- 使用同步数据库客户端
- 没有异步 API 需要调用

In [ ]:
# 方式二：只把阻塞调用放进线程
import asyncio


@app.get("/data")
async def get_data():
    result = await asyncio.to_thread(
        load_data
    )
    return result

适合：
- 路由整体使用异步代码
- 只有某一个旧客户端是同步的
- 需要同时组合异步 I/O 和同步 I/O

### 不要提前调用传给 to_thread 的函数
正确：
```
result = await asyncio.to_thread(
    sync_client.query,
    user_id,
)
```
- 函数对象：sync_client.query
- 函数参数：user_id

### 普通 def Dependency 也会进入线程池

### 线程适合 I/O 密集，不适合重 CPU 计算

#### 同步阻塞 I/O
```
等待同步数据库
等待同步 HTTP
等待磁盘
等待旧 SDK
```
因为线程等待期间，操作系统可以让其他线程运行。

#### CPU 密集
重 CPU 任务通常考虑：
- 进程池
- 独立任务 Worker
- Celery/RQ 等任务系统
- 专用推理服务
- 原生库提供的并行能力

### 线程池耗尽
现在有大量请求，每个同步路由都执行`time.sleep(30)`

一批请求占满线程池 → 新的同步任务无法马上获得线程 → 开始排队 → 延迟持续增长 → 请求可能超时

这叫资源饱和或线程池耗尽。

要分析根因：
- 是否可以换成真正异步客户端？
- 是否缺少数据库连接池限制？
- 是否应该增加超时？
- 是否应该限制并发？
- 是否需要将长任务放到任务队列？
- 是否需要拆分独立服务？
- 是否需要扩容 Worker？

## 面试
> FastAPI 会直接在事件循环中执行 async def 路由，而普通 def 路由和同步依赖会被放到外部线程池并异步等待。这样同步阻塞 I/O 不会直接卡住事件循环。但在异步路由中自己调用的普通辅助函数不会被自动转到线程池，如果它执行同步 I/O，需要使用 asyncio.to_thread() 或改用同步路由。线程池适合阻塞 I/O，重 CPU 任务通常应使用进程池、任务队列或独立服务。